In [17]:
import pandas as pd
from collections import defaultdict

In [18]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# show all columns
pd.set_option("display.max_columns", None)

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [20]:
%%R

require('tidyverse')
require('DescTools')

In [21]:
# Load match data

import glob

# Step 1: Get all CSV files in the folder
csv_files = glob.glob("/Users/hazelgandhi/Desktop/tennis-regression/files/*.csv")

# Step 2: Read and concatenate them
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Optional: Check the shape or preview
print(df.shape)
df.head()
df = df.sort_values('tourney_date')

five_set = ['Us Open','Roland Garros', 'Australian Open', 'Wimbledon']
df['five_set'] = df.tourney_name.isin(five_set)


# Initialize Elo ratings
elo_ratings = defaultdict(lambda: 1500)

# Store Elo snapshot *before* each match
elo_snapshots = []

K = 30

def win_prob(rating_i, rating_j):
    return 1 / (1 + 10 ** ((rating_j - rating_i) / 400))

for _, match in df.iterrows():
    winner = match['winner_name']
    loser = match['loser_name']

    rating_winner = elo_ratings[winner]
    rating_loser = elo_ratings[loser]

    # Record pre-match Elo ratings
    elo_snapshots.append({
        'date': match['tourney_date'],
        'surface': match['surface'],
        'tournament': match['tourney_name'],
        'winner': winner,
        'loser': loser,
        'five_set': match['five_set'],
        'winner_elo_before': rating_winner,
        'loser_elo_before': rating_loser
    })

    # Calculate expected outcomes
    expected_win = win_prob(rating_winner, rating_loser)
    expected_loss = 1 - expected_win

    # Update ratings
    elo_ratings[winner] += K * (1 - expected_win)
    elo_ratings[loser] += K * (0 - expected_loss)

# Create DataFrame
elo_df = pd.DataFrame(elo_snapshots)
elo_df

(5903, 49)


,date,surface,tournament,winner,loser,five_set,winner_elo_before,loser_elo_before
0,20220103,Hard,Atp Cup,Felix Auger Aliassime,Roberto Bautista Agut,False,1500.000000,1500.000000
1,20220103,Hard,Adelaide 1,Gianluca Mager,Francisco Cerundolo,False,1500.000000,1500.000000
2,20220103,Hard,Adelaide 1,Egor Gerasimov,Marton Fucsovics,False,1500.000000,1500.000000
3,20220103,Hard,Adelaide 1,Thiago Monteiro,Daniel Altmaier,False,1500.000000,1500.000000
4,20220103,Hard,Adelaide 1,Corentin Moutet,Holger Rune,False,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...
5898,20231127,Hard,NextGen Finals,Arthur Fils,Dominic Stricker,False,1594.446414,1549.788190
5899,20231127,Hard,NextGen Finals,Hamad Medjedovic,Dominic Stricker,False,1531.638967,1536.705692
5900,20231127,Hard,NextGen Finals,Arthur Fils,Luca Van Assche,False,1607.528912,1495.366613
5901,20231127,Hard,NextGen Finals,Hamad Medjedovic,Abedallah Shelbayh,False,1546.857700,1457.038529


In [22]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Randomly assign winner to be Player A or B
winner_is_A = np.random.randint(0, 2, size=len(elo_df)) == 1

# Create transformed DataFrame
elo_transformed = pd.DataFrame({
    'date': elo_df['date'],
    'tournament': elo_df['tournament'],
    'surface' : elo_df['surface'],
    'five_set': elo_df['five_set'],
    'player_A_name': np.where(winner_is_A, elo_df['winner'], elo_df['loser']),
    'player_B_name': np.where(winner_is_A, elo_df['loser'], elo_df['winner']),
    
    'player_A_elo_before': np.where(winner_is_A, elo_df['winner_elo_before'], elo_df['loser_elo_before']),
    'player_B_elo_before': np.where(winner_is_A, elo_df['loser_elo_before'], elo_df['winner_elo_before']),
    
    'player_A_win': winner_is_A.astype(int)
})
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win
0,20220103,Atp Cup,Hard,False,Roberto Bautista Agut,Felix Auger Aliassime,1500.000000,1500.000000,0
1,20220103,Adelaide 1,Hard,False,Gianluca Mager,Francisco Cerundolo,1500.000000,1500.000000,1
2,20220103,Adelaide 1,Hard,False,Marton Fucsovics,Egor Gerasimov,1500.000000,1500.000000,0
3,20220103,Adelaide 1,Hard,False,Daniel Altmaier,Thiago Monteiro,1500.000000,1500.000000,0
4,20220103,Adelaide 1,Hard,False,Holger Rune,Corentin Moutet,1500.000000,1500.000000,0
...,...,...,...,...,...,...,...,...,...
5898,20231127,NextGen Finals,Hard,False,Dominic Stricker,Arthur Fils,1549.788190,1594.446414,0
5899,20231127,NextGen Finals,Hard,False,Dominic Stricker,Hamad Medjedovic,1536.705692,1531.638967,0
5900,20231127,NextGen Finals,Hard,False,Arthur Fils,Luca Van Assche,1607.528912,1495.366613,1
5901,20231127,NextGen Finals,Hard,False,Hamad Medjedovic,Abedallah Shelbayh,1546.857700,1457.038529,1


In [23]:
elo_transformed['elo_difference'] = elo_transformed['player_A_elo_before'] - elo_transformed ['player_B_elo_before']
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference
0,20220103,Atp Cup,Hard,False,Roberto Bautista Agut,Felix Auger Aliassime,1500.000000,1500.000000,0,0.000000
1,20220103,Adelaide 1,Hard,False,Gianluca Mager,Francisco Cerundolo,1500.000000,1500.000000,1,0.000000
2,20220103,Adelaide 1,Hard,False,Marton Fucsovics,Egor Gerasimov,1500.000000,1500.000000,0,0.000000
3,20220103,Adelaide 1,Hard,False,Daniel Altmaier,Thiago Monteiro,1500.000000,1500.000000,0,0.000000
4,20220103,Adelaide 1,Hard,False,Holger Rune,Corentin Moutet,1500.000000,1500.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...
5898,20231127,NextGen Finals,Hard,False,Dominic Stricker,Arthur Fils,1549.788190,1594.446414,0,-44.658223
5899,20231127,NextGen Finals,Hard,False,Dominic Stricker,Hamad Medjedovic,1536.705692,1531.638967,0,5.066725
5900,20231127,NextGen Finals,Hard,False,Arthur Fils,Luca Van Assche,1607.528912,1495.366613,1,112.162299
5901,20231127,NextGen Finals,Hard,False,Hamad Medjedovic,Abedallah Shelbayh,1546.857700,1457.038529,1,89.819170


### Trying initial logistic model

In [24]:
%%R -i elo_transformed

logistic <- glm(player_A_win ~ elo_difference + elo_difference:five_set, data=elo_transformed, family=binomial)
print(summary(logistic))
print(PseudoR2(logistic, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set, 
    family = binomial, data = elo_transformed)

Coefficients:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 -0.0046813  0.0273657  -0.171    0.864    
elo_difference               0.0052695  0.0002990  17.624  < 2e-16 ***
elo_difference:five_setTRUE  0.0033810  0.0007439   4.545 5.49e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 8183.2  on 5902  degrees of freedom
Residual deviance: 7581.1  on 5900  degrees of freedom
AIC: 7587.1

Number of Fisher Scoring iterations: 4

 McFadden 
0.0735745 


### Adding surface clusters

In [25]:
surface_stats = elo_transformed.groupby(['player_A_name', 'surface'])['player_A_win'].agg(['sum', 'count']).reset_index()
surface_stats.columns = ['player', 'surface', 'wins', 'matches']
surface_stats['win_pct'] = surface_stats['wins'] / surface_stats['matches']

surface_stats

,player,surface,wins,matches,win_pct
0,Abedallah Shelbayh,Clay,1,1,1.000000
1,Abedallah Shelbayh,Hard,1,2,0.500000
2,Adrian Mannarino,Clay,2,7,0.285714
3,Adrian Mannarino,Grass,4,9,0.444444
4,Adrian Mannarino,Hard,30,46,0.652174
...,...,...,...,...,...
814,Zhizhen Zhang,Grass,1,3,0.333333
815,Zhizhen Zhang,Hard,4,11,0.363636
816,Zizou Bergs,Clay,1,1,1.000000
817,Zizou Bergs,Hard,1,5,0.200000


### Pivoting the table to wide format

In [26]:
surface_wide = surface_stats.pivot(index='player', columns='surface', values='win_pct').fillna(0)
surface_wide.columns = [f"{col}_win_pct" for col in surface_wide.columns]
surface_wide.reset_index(inplace=True)
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct
0,Abedallah Shelbayh,1.000000,0.000000,0.500000
1,Adrian Mannarino,0.285714,0.444444,0.652174
2,Ainius Sabaliauskas,0.000000,0.000000,1.000000
3,Aisam Ul Haq Qureshi,0.000000,1.000000,0.000000
4,Alan Fernando Rubio Fierros,0.000000,0.000000,0.000000
...,...,...,...,...
439,Zachary Svajda,0.000000,0.000000,0.250000
440,Zdenek Kolar,1.000000,0.000000,0.000000
441,Zhizhen Zhang,0.636364,0.333333,0.363636
442,Zizou Bergs,1.000000,0.000000,0.200000


In [27]:
## Adding clusters
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = surface_wide[['Clay_win_pct', 'Grass_win_pct', 'Hard_win_pct']]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

surface_wide['surface_cluster'] = clusters
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct,surface_cluster
0,Abedallah Shelbayh,1.000000,0.000000,0.500000,1
1,Adrian Mannarino,0.285714,0.444444,0.652174,3
2,Ainius Sabaliauskas,0.000000,0.000000,1.000000,2
3,Aisam Ul Haq Qureshi,0.000000,1.000000,0.000000,3
4,Alan Fernando Rubio Fierros,0.000000,0.000000,0.000000,0
...,...,...,...,...,...
439,Zachary Svajda,0.000000,0.000000,0.250000,0
440,Zdenek Kolar,1.000000,0.000000,0.000000,1
441,Zhizhen Zhang,0.636364,0.333333,0.363636,1
442,Zizou Bergs,1.000000,0.000000,0.200000,1


In [28]:
player_clusters = surface_wide[['player', 'surface_cluster']]


In [29]:
# For player A
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_A_name', right_on='player', how='left')
elo_transformed.rename(columns={'surface_cluster': 'surface_cluster_A'}, inplace=True)
elo_transformed.drop(columns='player', inplace=True)

# For player B
player_clusters.columns = ['player', 'surface_cluster_B']
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_B_name', right_on='player', how='left')
elo_transformed.drop(columns='player', inplace=True)
elo_transformed


,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference,surface_cluster_A,surface_cluster_B
0,20220103,Atp Cup,Hard,False,Roberto Bautista Agut,Felix Auger Aliassime,1500.000000,1500.000000,0,0.000000,3.0,3.0
1,20220103,Adelaide 1,Hard,False,Gianluca Mager,Francisco Cerundolo,1500.000000,1500.000000,1,0.000000,2.0,3.0
2,20220103,Adelaide 1,Hard,False,Marton Fucsovics,Egor Gerasimov,1500.000000,1500.000000,0,0.000000,3.0,0.0
3,20220103,Adelaide 1,Hard,False,Daniel Altmaier,Thiago Monteiro,1500.000000,1500.000000,0,0.000000,1.0,1.0
4,20220103,Adelaide 1,Hard,False,Holger Rune,Corentin Moutet,1500.000000,1500.000000,0,0.000000,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5898,20231127,NextGen Finals,Hard,False,Dominic Stricker,Arthur Fils,1549.788190,1594.446414,0,-44.658223,1.0,1.0
5899,20231127,NextGen Finals,Hard,False,Dominic Stricker,Hamad Medjedovic,1536.705692,1531.638967,0,5.066725,1.0,2.0
5900,20231127,NextGen Finals,Hard,False,Arthur Fils,Luca Van Assche,1607.528912,1495.366613,1,112.162299,1.0,1.0
5901,20231127,NextGen Finals,Hard,False,Hamad Medjedovic,Abedallah Shelbayh,1546.857700,1457.038529,1,89.819170,2.0,1.0


### Now the logistic model again

In [30]:
elo_transformed = elo_transformed.dropna()

In [31]:
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference,surface_cluster_A,surface_cluster_B
0,20220103,Atp Cup,Hard,False,Roberto Bautista Agut,Felix Auger Aliassime,1500.000000,1500.000000,0,0.000000,3.0,3.0
1,20220103,Adelaide 1,Hard,False,Gianluca Mager,Francisco Cerundolo,1500.000000,1500.000000,1,0.000000,2.0,3.0
2,20220103,Adelaide 1,Hard,False,Marton Fucsovics,Egor Gerasimov,1500.000000,1500.000000,0,0.000000,3.0,0.0
3,20220103,Adelaide 1,Hard,False,Daniel Altmaier,Thiago Monteiro,1500.000000,1500.000000,0,0.000000,1.0,1.0
4,20220103,Adelaide 1,Hard,False,Holger Rune,Corentin Moutet,1500.000000,1500.000000,0,0.000000,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5898,20231127,NextGen Finals,Hard,False,Dominic Stricker,Arthur Fils,1549.788190,1594.446414,0,-44.658223,1.0,1.0
5899,20231127,NextGen Finals,Hard,False,Dominic Stricker,Hamad Medjedovic,1536.705692,1531.638967,0,5.066725,1.0,2.0
5900,20231127,NextGen Finals,Hard,False,Arthur Fils,Luca Van Assche,1607.528912,1495.366613,1,112.162299,1.0,1.0
5901,20231127,NextGen Finals,Hard,False,Hamad Medjedovic,Abedallah Shelbayh,1546.857700,1457.038529,1,89.819170,2.0,1.0


In [32]:
%%R -i elo_transformed

logistic_new <- glm(player_A_win ~ elo_difference + elo_difference:five_set + surface_cluster_A + surface_cluster_B, data=elo_transformed, family=binomial)
print(summary(logistic_new))
print(PseudoR2(logistic_new, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set + 
    surface_cluster_A + surface_cluster_B, family = binomial, 
    data = elo_transformed)

Coefficients:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 -0.3810096  0.0855419  -4.454 8.43e-06 ***
elo_difference               0.0041912  0.0003130  13.391  < 2e-16 ***
surface_cluster_A            0.3333212  0.0283283  11.766  < 2e-16 ***
surface_cluster_B           -0.1758076  0.0287347  -6.118 9.46e-10 ***
elo_difference:five_setTRUE  0.0035103  0.0007484   4.691 2.72e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 7957.0  on 5740  degrees of freedom
Residual deviance: 7197.6  on 5736  degrees of freedom
AIC: 7207.6

Number of Fisher Scoring iterations: 4

  McFadden 
0.09543315 


### Checking prediction

In [33]:
%%R -i elo_transformed

df <- elo_transformed %>% mutate(
    predict_proba_R = predict(logistic_new, type="response"),
    predict_R = ifelse(predict_proba_R > .5, 1,0)
) %>% arrange(elo_difference)

df %>% head()

         date                      tournament surface five_set    player_A_name
5089 20230814              Cincinnati Masters    Hard    FALSE      Max Purcell
5270 20230828                         Us Open    Hard     TRUE Alexandre Muller
5881 20231123 Davis Cup Finals QF: SRB vs GBR    Hard    FALSE   Cameron Norrie
3827 20230320                   Miami Masters    Hard    FALSE   Facundo Bagnis
3594 20230227                           Dubai    Hard    FALSE     Tomas Machac
5251 20230828                         Us Open    Hard     TRUE  Dominik Koepfer
      player_B_name player_A_elo_before player_B_elo_before player_A_win
5089 Carlos Alcaraz            1407.245            1945.383            0
5270 Novak Djokovic            1496.437            1996.029            0
5881 Novak Djokovic            1552.741            2027.908            0
3827 Carlos Alcaraz            1393.393            1859.038            0
3594 Novak Djokovic            1478.222            1935.466            0
52

In [34]:
%%R

library(dplyr)

df_filtered <- df %>%
  filter(player_A_win != predict_R) %>%
  arrange(desc(predict_proba_R))

head(df_filtered)

         date     tournament surface five_set         player_A_name
5271 20230828        Us Open    Hard     TRUE           Holger Rune
4250 20230508   Rome Masters    Clay    FALSE        Carlos Alcaraz
4395 20230529  Roland Garros    Clay     TRUE Felix Auger Aliassime
2987 20230102     Adelaide 1    Hard    FALSE Felix Auger Aliassime
4121 20230424 Madrid Masters    Clay    FALSE       Daniil Medvedev
4333 20230529  Roland Garros    Clay     TRUE       Sebastian Korda
               player_B_name player_A_elo_before player_B_elo_before
5271 Roberto Carballes Baena            1771.490            1511.918
4250         Fabian Marozsan            1920.819            1509.091
4395           Fabio Fognini            1666.622            1455.023
2987          Alexei Popyrin            1776.503            1391.430
4121          Aslan Karatsev            1883.343            1437.913
4333         Sebastian Ofner            1665.560            1479.325
     player_A_win elo_difference surface_

In [35]:
%%R 

# Create confusion matrix
conf_mat <- table(df$predict_R, df$player_A_win)
print(conf_mat) 

# Extract TP, FP, FN
TP <- conf_mat[2,2]
FP <- conf_mat[2,1]
FN <- conf_mat[1,2]
# Calculate Precision and Recall
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
# Print
cat("Precision:", precision, "\n")
cat("Recall:", recall, "\n")

   
       0    1
  0 1910 1018
  1 1010 1803
Precision: 0.6409527 
Recall: 0.6391351 


In [36]:
%%R
nrow(df_filtered)


[1] 2028


In [37]:
%%R
write.csv(df, "final-analysis-df.csv", row.names = FALSE)